# 🤖 Train Vision Transformer (ViT) on CIFAKE Dataset (120k Images)

This notebook contains the complete, executable pipeline to fine-tune a pre-trained Vision Transformer (`google/vit-base-patch16-224-in21k`) on the 120,000-image **CIFAKE dataset (Real vs AI-Generated)**.

**Hardware Requirements:**
- **Google Colab:** Free Tier T4 GPU is supported
- **Runtime:** Go to `Runtime > Change runtime type > Hardware accelerator > T4 GPU`.

*Training 120,000 images over 5 epochs takes approximately 2.5 - 3 hours on a single T4 GPU. We will stream the dataset from Kaggle instead of downloading the 100MB+ zip to save disk space and loading time.*

In [ ]:
!pip install -q transformers datasets evaluate accelerate torch torchvision

import torch
print("Hardware details:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Step 1: Initialize Environment
We use `datasets` for streaming the images and `transformers` for the ViT.

In [ ]:
import os
os.makedirs("best_model", exist_ok=True)

## Step 2: Load the CIFAKE Dataset
We load the 120,000 image dataset directly from the Kaggle repository.

In [ ]:
from datasets import load_dataset

# The dataset comes pre-split into 100k train and 20k test images
dataset = load_dataset("dragonintelligence/CIFAKE-image-dataset")
print(dataset)

# Extract labels list
labels = dataset["train"].features["label"].names
label2id = {label: str(i) for i, label in enumerate(labels)}
id2label = {str(i): label for i, label in enumerate(labels)}

## Step 3: Image Preprocessing (Transforms)
We use `ViTImageProcessor` to convert PIL images into pixel values exactly how the pre-trained ViT expects them (normalized, 224x224).

In [ ]:
from transformers import ViTImageProcessor
from torchvision.transforms import RandomResizedCrop, RandomHorizontalFlip, ToTensor, Normalize, Compose, Resize, CenterCrop

model_checkpoint = "google/vit-base-patch16-224-in21k"
image_processor = ViTImageProcessor.from_pretrained(model_checkpoint)

# Mean & Std defaults for ViT
normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)

train_transforms = Compose([
    RandomResizedCrop(image_processor.size["height"]),
    RandomHorizontalFlip(),
    ToTensor(),
    normalize,
])

val_transforms = Compose([
    Resize(image_processor.size["height"]),
    CenterCrop(image_processor.size["height"]),
    ToTensor(),
    normalize,
])

def preprocess_train(example_batch):
    example_batch["pixel_values"] = [train_transforms(image.convert("RGB")) for image in example_batch["image"]]
    return example_batch

def preprocess_val(example_batch):
    example_batch["pixel_values"] = [val_transforms(image.convert("RGB")) for image in example_batch["image"]]
    return example_batch

train_ds = dataset['train'].with_transform(preprocess_train)
val_ds = dataset['test'].with_transform(preprocess_val)

## Step 4: Define Evaluation Metrics

In [ ]:
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

## Step 5: Load Pre-trained Vision Transformer

In [ ]:
from transformers import ViTForImageClassification

model = ViTForImageClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

## Step 6: Initialize Trainer & Start Training
We use mixed precision (`fp16`) to drastically reduce GPU VRAM usage and speed up training.

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="ai_vs_real_image_detection",
    remove_unused_columns=False,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=64, 
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    warmup_ratio=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False, # We will download the zip manually at the end
    fp16=torch.cuda.is_available(), 
)

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

trainer = Trainer(
    model,
    args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=image_processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

print("🚀 Starting Training on 100,000 images! Go get a coffee... ☕")
trainer.train()

## Step 7: Final Evaluation & Download Model
We evaluate the best model, save it locally, zip it, and download it exactly as you did in Colab.

In [ ]:
trainer.evaluate()

print("Saving model to disk...")
trainer.save_model("best_model")

import shutil
from google.colab import files

print("Zipping folder for download...")
shutil.make_archive("best_model", 'zip', "best_model")

print("✅ Training Complete. Downloading best_model.zip to local machine...")
files.download("best_model.zip")